###                    END - END - ANN - PROJECT --- USING MEDICAL INSURANCE DATASET
### Predicting High-Risk Patients (`is_high_risk`) with an Artificial Neural Network

## Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

from keras.models import Sequential
from keras.layers import Input, Dense, Dropout, BatchNormalization
from keras.regularizers import l1_l2
from keras.optimizers import Adam, RMSprop, SGD
from keras.callbacks import EarlyStopping

from imblearn.over_sampling import SMOTE

# pip install optuna -q
import optuna

import pickle

pd.set_option("display.max_columns", None)


## Load Dataset

In [ ]:
# If running in Colab, upload medical_insurance.csv to /content/ first (or mount Drive).
df = pd.read_csv(r"/content/medical_insurance.csv")
df.head()


## Explore Dataset (Basic checks)

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.describe()
df.groupby("is_high_risk")[["age", "chronic_count", "risk_score", "annual_medical_cost"]].mean()
df[["risk_score", "is_high_risk"]].corr()


In [ ]:
df.isnull().sum()

In [ ]:
df.duplicated().sum()

## Handle Missing Values

`alcohol_freq` has ~30k missing values. There is no explicit `None` category recorded for it, so a missing entry is treated as "no alcohol use reported" and filled with `"None"`.

In [ ]:
df["alcohol_freq"] = df["alcohol_freq"].fillna("None")
df.isnull().sum().sum()   # should be 0 now


## Drop Duplicate Rows

In [ ]:
df = df.drop_duplicates()
df.duplicated().sum()


## Handle Data Leakage

`risk_score` correlates ~0.82 with `is_high_risk` — it is almost certainly the score the label was derived from, so it is dropped to avoid leaking the answer into the model. `person_id` is just an identifier and carries no predictive signal, so it is dropped too.

In [ ]:
df = df.drop(columns=["risk_score", "person_id"])
df.shape


## Separate Features and Target

In [ ]:
X = df.drop("is_high_risk", axis=1)
y = df["is_high_risk"]

y.value_counts(normalize=True)


## Identify Numeric & Categorical Columns

In [ ]:
num_cols = X.select_dtypes(exclude="object").columns.tolist()
cat_cols = X.select_dtypes(include="object").columns.tolist()

print("Numeric columns:", len(num_cols))
print("Categorical columns:", cat_cols)


## Train / Validation / Test Split

In [ ]:
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.2, random_state=21, stratify=y
)


In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.2, random_state=21, stratify=y_train_full
)

X_train.shape, X_val.shape, X_test.shape


## Feature Scaling & Encoding

Numeric columns are standardized with `StandardScaler` and categorical columns are one-hot encoded with `OneHotEncoder`, combined via a `ColumnTransformer`. The transformer is **fit only on the training set** and reused to transform validation/test to avoid leakage.

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ]
)

X_train_t = preprocessor.fit_transform(X_train)
X_val_t = preprocessor.transform(X_val)
X_test_t = preprocessor.transform(X_test)

# OneHotEncoder can return a sparse matrix - densify for Keras
if hasattr(X_train_t, "toarray"):
    X_train_t = X_train_t.toarray()
    X_val_t = X_val_t.toarray()
    X_test_t = X_test_t.toarray()

X_train_t.shape


## Build ANN (Baseline Model)

In [ ]:
model = Sequential([
    Input(shape=(X_train_t.shape[1],)),
    Dense(64, activation="relu"),
    Dense(32, activation="relu"),
    Dense(1, activation="sigmoid"),
])

model.summary()


## Compile Model

In [ ]:
model.compile(optimizer="Adam", loss="binary_crossentropy", metrics=["accuracy"])


In [ ]:
early_stop = EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)


In [ ]:
model.fit(
    X_train_t, y_train,
    epochs=50,
    validation_data=(X_val_t, y_val),
    batch_size=32,
    callbacks=[early_stop],
    verbose=2,
)


In [ ]:
y_pred = np.where(model.predict(X_test_t) > 0.5, 1, 0)
accuracy_score(y_test, y_pred)


## Handle Class Imbalance (SMOTE)

`is_high_risk` is imbalanced (~37% positive). SMOTE oversamples the minority class on the **training** set only.

In [ ]:
smote = SMOTE(random_state=21)
X_train_res, y_train_res = smote.fit_resample(X_train_t, y_train)

y_train_res.value_counts()


## Hyperparameter Tuning (Optuna)

In [ ]:
def objective(trial):
    lr_rate = trial.suggest_float("learning_rate", 1e-4, 1e-2, log=True)
    n_layers = trial.suggest_int("n_layers", 1, 4)
    optimizer_name = trial.suggest_categorical("optimizer", ["Adam", "RMSprop", "SGD"])
    activation = trial.suggest_categorical("activation", ["tanh", "relu"])
    batch_size1 = trial.suggest_categorical("batch_size", [32, 64, 128, 256, 512])

    model = Sequential()
    model.add(Input(shape=(X_train_res.shape[1],)))
    for i in range(n_layers):
        units = trial.suggest_int(f"units{i}", 8, 96)
        dropout = trial.suggest_float(f"dropout{i}", 0.0, 0.5)
        reg = trial.suggest_float(f"reg{i}", 1e-5, 1e-2, log=True)
        model.add(Dense(units, activation=activation, kernel_regularizer=l1_l2(l1=reg, l2=reg)))
        model.add(BatchNormalization())
        model.add(Dropout(dropout))
    model.add(Dense(1, activation="sigmoid"))

    if optimizer_name == "Adam":
        opt = Adam(learning_rate=lr_rate)
    elif optimizer_name == "RMSprop":
        opt = RMSprop(learning_rate=lr_rate)
    else:
        opt = SGD(learning_rate=lr_rate)

    model.compile(optimizer=opt, loss="binary_crossentropy", metrics=["accuracy"])

    es = EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)
    history = model.fit(
        X_train_res, y_train_res,
        epochs=20,
        batch_size=batch_size1,
        validation_data=(X_val_t, y_val),
        callbacks=[es],
        verbose=0,
    )
    return min(history.history["val_loss"])


In [ ]:
study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=21))
study.optimize(objective, n_trials=15, show_progress_bar=True)


In [ ]:
study.best_params

## Rebuilding the Model with Best Hyperparameters

Instead of hand-copying the winning trial's numbers, the final architecture is built programmatically straight from `study.best_params`, so re-running the tuning cell automatically carries through to the final model.

In [ ]:
def build_best_model(params, input_dim):
    model = Sequential()
    model.add(Input(shape=(input_dim,)))
    for i in range(params["n_layers"]):
        model.add(Dense(
            params[f"units{i}"],
            activation=params["activation"],
            kernel_initializer="he_normal",
            kernel_regularizer=l1_l2(l1=params[f"reg{i}"], l2=params[f"reg{i}"]),
        ))
        model.add(BatchNormalization())
        model.add(Dropout(params[f"dropout{i}"]))
    model.add(Dense(1, activation="sigmoid", kernel_initializer="glorot_uniform"))

    if params["optimizer"] == "Adam":
        opt = Adam(learning_rate=params["learning_rate"])
    elif params["optimizer"] == "RMSprop":
        opt = RMSprop(learning_rate=params["learning_rate"])
    else:
        opt = SGD(learning_rate=params["learning_rate"])

    model.compile(
        optimizer=opt,
        loss="binary_crossentropy",
        metrics=["accuracy", tf.keras.metrics.Recall(name="recall")],
    )
    return model

best_params = study.best_params
model = build_best_model(best_params, X_train_res.shape[1])
model.summary()


In [ ]:
es = EarlyStopping(monitor="val_loss", mode="min", patience=5, restore_best_weights=True)

history = model.fit(
    X_train_res, y_train_res,
    epochs=50,
    batch_size=best_params["batch_size"],
    validation_data=(X_val_t, y_val),   # validation set, NOT test
    callbacks=[es],
    verbose=0,
)


In [ ]:
y_pred = np.where(model.predict(X_test_t) > 0.5, 1, 0)
accuracy_score(y_test, y_pred)


## Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history.history["accuracy"], label="train")
axes[0].plot(history.history["val_accuracy"], label="val")
axes[0].set_title("Accuracy"); axes[0].legend()

axes[1].plot(history.history["recall"], label="train")
axes[1].plot(history.history["val_recall"], label="val")
axes[1].set_title("Recall"); axes[1].legend()
plt.tight_layout()
plt.show()


## Train / Validation / Test Accuracy Comparison

In [ ]:
y_pred_train = np.where(model.predict(X_train_res) > 0.5, 1, 0)
train_acc = accuracy_score(y_train_res, y_pred_train)

y_pred_val = np.where(model.predict(X_val_t) > 0.5, 1, 0)
val_acc = accuracy_score(y_val, y_pred_val)

y_pred_test = np.where(model.predict(X_test_t) > 0.5, 1, 0)
test_acc = accuracy_score(y_test, y_pred_test)

print(f"Train Accuracy      : {train_acc:.4f}")
print(f"Validation Accuracy : {val_acc:.4f}")
print(f"Test Accuracy       : {test_acc:.4f}")

gap = abs(train_acc - test_acc)
print(f"\nTrain/Test gap: {gap:.4f} -> {'Good fit' if gap < 0.05 else 'Possible overfit, revisit regularization'}")


## Classification Report & Confusion Matrix

In [ ]:
print(classification_report(y_test, y_pred_test))

In [ ]:
cm = confusion_matrix(y_test, y_pred_test)
plt.figure(figsize=(4, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False,
            xticklabels=["Low Risk", "High Risk"], yticklabels=["Low Risk", "High Risk"])
plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.title("Confusion Matrix")
plt.show()


## Save Model & Preprocessing Artifacts

The trained model plus the fitted `ColumnTransformer` and training columns are saved so a new patient row can be preprocessed and scored the same way at inference time.

In [ ]:
model.save("medical_insurance_ann.keras")

with open("preprocessor.pkl", "wb") as f:
    pickle.dump(preprocessor, f)

training_columns = X.columns.tolist()
with open("training_columns.pkl", "wb") as f:
    pickle.dump(training_columns, f)

with open("best_hyperparameters.pkl", "wb") as f:
    pickle.dump(best_params, f)

print("Saved model, preprocessor, training columns, and best hyperparameters.")
